# Kriging with the Vecchia log-likelihood — objective="VLL(m)" (R)

This notebook demonstrates the **Vecchia approximated log-likelihood** as a fit
objective. Vecchia (1988): log p(y) is approximated by the product of the
conditionals of each observation given its $m$ nearest previously-ordered
neighbors (greedy maxmin ordering, Guinness 2018):

$$\log L \approx \sum_i \log p(y_i \mid y_{N(i)}), \qquad |N(i)| \le m$$

Each evaluation costs $O(n\,m^3)$ instead of $O(n^3)$, is a valid Gaussian
density (sparse inverse Cholesky), and is exact for $m = n-1$. The usage could
not be simpler — it is just an `objective` string:

- `objective="VLL"` (default $m=30$) or `objective="VLL(m)"`.

After the fit, ONE exact factorization is performed at $\theta^*$, so
`predict`/`simulate`/`update` behave exactly as after an `"LL"` fit.

Steps:
1. Setup
2. Branin function
3. Design of experiments ($n = 1500$)
4. Fit with `"VLL(30)"` vs exact `"LL"`: wall time and hyperparameters
5. Predictions: both models on a grid, RMSE
6. Effect of the number of neighbors $m$


## 0. Installation (run once)

Build the C++ core and the R binding from source.
Requires: `cmake`, a C++ compiler, and R development headers.

The script `tools/r-linux-macos/build.sh` calls `tools/linux-macos/build.sh`
to build the C++ core, then runs `make` in `bindings/R` to compile and install
**rlibkriging** into `bindings/R/Rlibs`.

In [ ]:
# Run this cell once to build and install rlibkriging.
# Skip if already built (bindings/R/Rlibs/rlibkriging exists).
repo_root <- normalizePath(file.path(getwd(), "../.."), mustWork = FALSE)
rlibs     <- file.path(repo_root, "bindings", "R", "Rlibs", "rlibkriging")

if (!dir.exists(rlibs)) {
  message("Building rlibkriging from source…")
  ret <- system(paste0("cd '", repo_root, "' && bash tools/r-linux-macos/build.sh"))
  if (ret != 0) stop("Build failed — check compiler and cmake installation.")
} else {
  message("rlibkriging already built, skipping.")
}

## 1. Load rlibkriging

In [ ]:
repo_root <- normalizePath(file.path(getwd(), "../.."), mustWork = FALSE)
lib_path  <- file.path(repo_root, "bindings", "R", "Rlibs")
library(rlibkriging, lib.loc = lib_path)

## 2. Branin function

In [ ]:
branin <- function(X) {
  x1 <- X[, 1] * 15 - 5
  x2 <- X[, 2] * 15
  (x2 - 5 / (4 * pi^2) * x1^2 + 5 / pi * x1 - 6)^2 +
    10 * (1 - 1 / (8 * pi)) * cos(x1) + 10
}
grid_x <- seq(0, 1, length.out = 50)
grid_pts <- as.matrix(expand.grid(grid_x, grid_x))
z_true <- matrix(branin(grid_pts), 50, 50)
filled.contour(grid_x, grid_x, z_true, main = "True Branin")

## 3. Design of experiments ($n = 1500$)

In [ ]:
set.seed(123)
n <- 1500
X <- matrix(runif(2 * n), ncol = 2)
y <- branin(X)

## 4. Fit: `VLL(30)` vs exact `LL`

In [ ]:
t_vll <- system.time(k_vll <- Kriging(y, X, kernel = "matern5_2", objective = "VLL(30)"))["elapsed"]
t_ll  <- system.time(k_ll  <- Kriging(y, X, kernel = "matern5_2", objective = "LL"))["elapsed"]
cat(sprintf("fit VLL(30) : %6.1f s   theta = %s\n", t_vll, paste(round(k_vll$theta(), 4), collapse = " ")))
cat(sprintf("fit LL      : %6.1f s   theta = %s\n", t_ll, paste(round(k_ll$theta(), 4), collapse = " ")))
cat(sprintf("speedup     : %.1fx\n", t_ll / t_vll))

## 5. Predictions

In [ ]:
p_vll <- predict(k_vll, grid_pts, return_stdev = FALSE)
p_ll  <- predict(k_ll, grid_pts, return_stdev = FALSE)
filled.contour(grid_x, grid_x, matrix(p_vll$mean, 50, 50), main = "mean — fit VLL(30)")
cat("RMSE vs true (VLL):", sqrt(mean((p_vll$mean - as.vector(z_true))^2)), "\n")
cat("RMSE vs true (LL) :", sqrt(mean((p_ll$mean - as.vector(z_true))^2)), "\n")
cat("max |VLL - LL|    :", max(abs(p_vll$mean - p_ll$mean)), "\n")

## 6. Effect of the number of neighbors $m$

In [ ]:
for (m in c(5, 15, 30)) {
  t <- system.time(k <- Kriging(y, X, kernel = "matern5_2", objective = sprintf("VLL(%d)", m)))["elapsed"]
  p <- predict(k, grid_pts, return_stdev = FALSE)
  rmse <- sqrt(mean((p$mean - as.vector(z_true))^2))
  cat(sprintf("VLL(%2d) : fit %6.1f s   theta = %s   RMSE = %.4f\n",
              m, t, paste(round(k$theta(), 4), collapse = " "), rmse))
}

## Notes

- The screening effect behind Vecchia weakens in high dimension: recommended
  for $d \le 5$ (complementary to `NestedKriging`, which is dimension-robust).
- The final exact factorization at $\theta^*$ still costs $O(n^3)$ once. For
  very large $n$, the C++ API offers `predictVecchia` (local $O(q\,m^3)$
  prediction) and a "light" mode (`set_vecchia_exact_commit(false)`) that
  skips it entirely — bindings for both are planned.
- References: Vecchia (1988); Guinness (2018), *Technometrics*; Katzfuss &
  Guinness (2021), *Statistical Science*.
